# 01 · SQL Basics and the Order a Query Runs In

Most company data does not live in CSV files. It lives in **databases**, and the language to ask a database questions is **SQL** (Structured Query Language). Pandas is great once the data is on your laptop. SQL is how you get the right data there in the first place, and often how you do the whole analysis.

**What you will learn here**

1. Why SQLite, and how to run SQL from Python
2. `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`
3. `NULL`: the missing value, and the traps it creates
4. `CASE WHEN`: if/else inside a query
5. `GROUP BY` and `HAVING`: summaries per group
6. The **logical order** a query runs in, executed step by step on 10 passengers

> **Colour guide:** $\color{red}{\text{red}}$ marks a warning, a trap or a wrong result. $\color{green}{\text{green}}$ marks a passed check or a correct result.

## Why SQLite?

There are many databases. The SQL core (`SELECT`, `JOIN`, `GROUP BY`, window functions) is almost the same in all of them.

| Database | Setup | Typical use |
|---|---|---|
| **SQLite** | None: built into Python, the whole database is one file | Apps, phones, local analysis, learning |
| **PostgreSQL** | Install and run a server | Most common open-source database for company data |
| **MySQL** | Install and run a server | Web applications |
| **BigQuery, Snowflake** | Cloud account | Large data warehouses |

We use **SQLite** because anyone who clones this repo can run every query with no installation and no login. Everything in this folder also works in PostgreSQL, and where the two behave differently, the notebook says so.

**How it works in Python:** the `sqlite3` module (part of Python) opens a database, and `pandas.read_sql_query` runs a query and returns the result as a table.

## The dataset: Titanic

We start with a table we already know from the statistics folder, so we can focus on the SQL. Source: [seaborn-data](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv), the same 891 passengers as the Kaggle Titanic training set (historical records, free for teaching). Saved in `data/titanic.csv`.

| Column | Meaning |
|---|---|
| `survived` | 1 = survived, 0 = died |
| `pclass` | Ticket class: 1, 2 or 3 |
| `sex` | female / male |
| `age` | Age in years (missing for some passengers) |
| `fare` | Ticket price |
| `embark_town` | Port where the passenger boarded |
| `deck` | Deck of the cabin (often missing) |

We copy the CSV into a SQLite database that lives in memory, as a table called `passengers`.

In [1]:
import sqlite3
import pandas as pd

# docs: https://docs.python.org/3/library/sqlite3.html
con = sqlite3.connect(":memory:")   # a temporary database in memory

titanic = pd.read_csv("../data/titanic.csv")
titanic.to_sql("passengers", con, index=False)

def run(query):
    # docs: https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html
    return pd.read_sql_query(query, con)

print("SQLite version:", sqlite3.sqlite_version)
run("SELECT COUNT(*) AS rows FROM passengers")

SQLite version: 3.45.3


,rows
0,891


---
## 1. SELECT: choose the columns

> **Theory.** A table has **rows** (one per passenger) and **columns** (one per piece of information). `SELECT` picks the columns, `FROM` says which table.

**Syntax**

```sql
SELECT column1, column2      -- which columns
FROM   table_name            -- which table
LIMIT  5;                    -- only the first 5 rows
```

- `SELECT *` means "all columns".
- `AS` gives a column a new name (an **alias**).
- You can compute new columns: `fare * 2 AS double_fare`.
- SQL keywords are not case-sensitive. Writing them in CAPITALS is just a habit that makes queries easier to read.

In [2]:
run('''
SELECT sex, age, fare, ROUND(fare / 10, 1) AS fare_in_tens
FROM passengers
LIMIT 5
''')

,sex,age,fare,fare_in_tens
0,male,22.0,7.2500,0.7
1,female,38.0,71.2833,7.1
2,female,26.0,7.9250,0.8
3,female,35.0,53.1000,5.3
4,male,35.0,8.0500,0.8


---
## 2. WHERE: choose the rows

> **Theory.** `WHERE` keeps only the rows where a condition is true.

| Operator | Meaning | Example |
|---|---|---|
| `=`, `<>` (or `!=`) | equal, not equal | `sex = 'female'` |
| `<`, `>`, `<=`, `>=` | comparisons | `age < 18` |
| `AND`, `OR`, `NOT` | combine conditions | `age < 18 AND sex = 'male'` |
| `IN (...)` | one of a list | `pclass IN (1, 2)` |
| `BETWEEN a AND b` | in a range (both ends included) | `fare BETWEEN 10 AND 20` |
| `LIKE` | text pattern: `%` = any text, `_` = one character | `embark_town LIKE 'S%'` |

Text values go in **single quotes**: `'female'`.

In [3]:
run('''
SELECT sex, pclass, age, fare
FROM passengers
WHERE sex = 'female' AND pclass IN (1, 2) AND age BETWEEN 20 AND 25
LIMIT 5
''')

,sex,pclass,age,fare
0,female,2,21.0,10.5
1,female,1,23.0,263.0
2,female,1,22.0,66.6
3,female,2,24.0,13.0
4,female,2,24.0,14.5


### Trap: AND is evaluated before OR

**Question:** how many women travelled in first **or** second class?

In [4]:
without_brackets = run('''
SELECT COUNT(*) AS n FROM passengers
WHERE sex = 'female' AND pclass = 1 OR pclass = 2
''')
with_brackets = run('''
SELECT COUNT(*) AS n FROM passengers
WHERE sex = 'female' AND (pclass = 1 OR pclass = 2)
''')
print("without brackets:", without_brackets["n"][0])
print("with brackets   :", with_brackets["n"][0])

without brackets: 278
with brackets   : 170


$\color{red}{\text{Without brackets the answer is wrong (278).}}$ SQL reads it as `(female AND first class) OR (second class)`, so it also counts every **man** in second class. $\color{green}{\text{With brackets we get the right answer (170).}}$

> **Key idea:** whenever you mix `AND` and `OR`, add brackets.

---
## 3. ORDER BY, LIMIT and DISTINCT

- `ORDER BY column` sorts the result, `ASC` (smallest first, the default) or `DESC` (largest first).
- `LIMIT n` keeps only the first n rows. Together they answer "top n" questions.
- `DISTINCT` removes duplicate rows.

In [5]:
print("The 3 most expensive tickets:")
display(run('''
SELECT sex, pclass, age, fare
FROM passengers
ORDER BY fare DESC
LIMIT 3
'''))

print("Every port passengers boarded at:")
display(run("SELECT DISTINCT embark_town FROM passengers"))

The 3 most expensive tickets:


,sex,pclass,age,fare
0,female,1,35.0,512.3292
1,male,1,36.0,512.3292
2,male,1,35.0,512.3292


Every port passengers boarded at:


,embark_town
0,Southampton
1,Cherbourg
2,Queenstown
3,None


Notice `None` in the list of ports: two passengers have no boarding port. That is a `NULL`, the subject of the next section.

---
## 4. NULL: the missing value

> **Theory.** `NULL` means "unknown". It is not 0 and not an empty text. Because it is unknown, **any comparison with NULL gives "unknown"**, which `WHERE` treats as false:
>
> - `NULL = 30` → unknown, `NULL <> 30` → unknown, even `NULL = NULL` → unknown
> - To find missing values you must write `IS NULL` or `IS NOT NULL`

| Function | What it does with NULL |
|---|---|
| `COUNT(*)` | counts **all** rows |
| `COUNT(column)` | counts only rows where the column is **not** NULL |
| `AVG`, `SUM`, `MIN`, `MAX` | **ignore** NULL values |
| `COALESCE(a, b)` | returns `a`, or `b` if `a` is NULL |

**Small example.** 4 ages: 20, 40, NULL, 30.

- `COUNT(*)` = 4, `COUNT(age)` = 3
- `AVG(age)` = (20 + 40 + 30) / 3 = **30** (the NULL is skipped, not counted as 0)
- `SUM(age) / COUNT(*)` = 90 / 4 = **22.5** (wrong: treats the unknown age as if it were 0)

In [6]:
run('''
SELECT COUNT(*)                     AS all_rows,
       COUNT(age)                   AS rows_with_age,
       COUNT(deck)                  AS rows_with_deck,
       ROUND(AVG(age), 2)           AS avg_age,
       ROUND(SUM(age) / COUNT(*), 2) AS wrong_avg_age
FROM passengers
''')

,all_rows,rows_with_age,rows_with_deck,avg_age,wrong_avg_age
0,891,714,203,29.7,23.8


177 passengers have no age and 688 have no deck. `AVG(age)` = 29.70 is the average of the known ages. $\color{red}{\text{Dividing by COUNT(*) gives 23.80}}$, as if every unknown passenger were a newborn.

### Trap: NULL rows silently disappear from "not equal"

In [7]:
run('''
SELECT (SELECT COUNT(*) FROM passengers)                 AS total,
       (SELECT COUNT(*) FROM passengers WHERE age = 30)  AS age_is_30,
       (SELECT COUNT(*) FROM passengers WHERE age <> 30) AS age_is_not_30,
       (SELECT COUNT(*) FROM passengers WHERE age IS NULL) AS age_unknown
''')

,total,age_is_30,age_is_not_30,age_unknown
0,891,25,689,177


25 + 689 = 714, not 891. $\color{red}{\text{The 177 passengers with an unknown age are in neither group}}$, because `NULL <> 30` is "unknown", not true. If you want them, say so: `WHERE age <> 30 OR age IS NULL`.

> **Key idea:** before filtering or averaging a column, check how many NULLs it has.

---
## 5. CASE WHEN: if/else inside a query

> **Theory.** `CASE` creates a new value from conditions, checked from top to bottom. The first true condition wins. `ELSE` catches everything else, including NULL.

```sql
CASE WHEN condition1 THEN value1
     WHEN condition2 THEN value2
     ELSE other_value
END
```

In [8]:
run('''
SELECT CASE WHEN age < 18 THEN 'child'
            WHEN age < 60 THEN 'adult'
            WHEN age >= 60 THEN 'senior'
            ELSE 'unknown'
       END AS age_group,
       COUNT(*) AS passengers
FROM passengers
GROUP BY age_group
ORDER BY passengers DESC
''')

,age_group,passengers
0,adult,575
1,unknown,177
2,child,113
3,senior,26


Without the `ELSE 'unknown'` line, the 177 passengers with no age would end up in a group called NULL. Writing it out makes the missing data visible.

---
## 6. GROUP BY and HAVING

> **Theory.** `GROUP BY` puts rows with the same value into one group, then an **aggregate function** (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) turns each group into one row. It is the SQL version of `df.groupby(...).agg(...)` in pandas.
>
> - `WHERE` filters **rows**, before grouping
> - `HAVING` filters **groups**, after grouping

Rule: every column in `SELECT` must be either in the `GROUP BY` or inside an aggregate function.

**Question:** what was the survival rate for each sex and class, for groups of more than 100 passengers?

In [9]:
run('''
SELECT sex,
       pclass,
       COUNT(*)                       AS passengers,
       ROUND(100.0 * AVG(survived), 1) AS survival_rate_pct
FROM passengers
GROUP BY sex, pclass
HAVING COUNT(*) > 100
ORDER BY survival_rate_pct DESC
''')

,sex,pclass,passengers,survival_rate_pct
0,female,3,144,50.0
1,male,1,122,36.9
2,male,2,108,15.7
3,male,3,347,13.5


Only 4 of the 6 groups have more than 100 passengers. Men in third class, the largest group (347), had the lowest survival rate: 13.5%.

### Trap: integer division

`survived` holds whole numbers (0 and 1). In SQLite and PostgreSQL, **whole number ÷ whole number = whole number**, rounded down.

In [10]:
run('''
SELECT SUM(survived) / COUNT(*)       AS integer_division,
       1.0 * SUM(survived) / COUNT(*) AS decimal_division,
       AVG(survived)                  AS with_avg
FROM passengers
''')

,integer_division,decimal_division,with_avg
0,0,0.383838,0.383838


$\color{red}{\text{342 / 891 gives 0}}$: the survival rate looks like zero. $\color{green}{\text{Multiplying by 1.0 first, or using AVG, gives 0.384.}}$ That's why the query above used `100.0 * AVG(...)`.

---
## 7. The order a query really runs in

> **Theory.** We **write** a query in this order: `SELECT … FROM … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`. But the database **runs** it in a different, logical order:

| Step | Clause | What it does |
|---|---|---|
| 1 | `FROM` (and `JOIN`) | take the table(s) |
| 2 | `WHERE` | keep the rows that match |
| 3 | `GROUP BY` | put rows into groups |
| 4 | `HAVING` | keep the groups that match |
| 5 | `SELECT` | compute the output columns and aliases |
| 6 | `DISTINCT` | remove duplicate rows |
| 7 | `ORDER BY` | sort |
| 8 | `LIMIT` | keep the first n rows |

This order explains most "why does my query fail?" moments:

- `WHERE` runs before grouping, so it **cannot** use `COUNT(*)`. That's what `HAVING` is for.
- `SELECT` runs after `WHERE`, so in standard SQL (and PostgreSQL) a `WHERE` **cannot** use an alias made in `SELECT`. (SQLite is lenient and allows it, but don't rely on it.)
- `ORDER BY` runs after `SELECT`, so it **can** use the aliases.

### Step-by-step execution on 10 passengers

**Query:** for each class, how many passengers with a known age, and their average fare? Keep classes with at least 2 such passengers, show the most expensive one.

```sql
SELECT pclass, COUNT(*) AS passengers, ROUND(AVG(fare), 2) AS avg_fare
FROM sample
WHERE age IS NOT NULL
GROUP BY pclass
HAVING COUNT(*) >= 2
ORDER BY avg_fare DESC
LIMIT 1;
```

**Step 1, FROM:** the first 10 passengers.

| row | pclass | age | fare |
|---|---|---|---|
| 0 | 3 | 22 | 7.25 |
| 1 | 1 | 38 | 71.28 |
| 2 | 3 | 26 | 7.93 |
| 3 | 1 | 35 | 53.10 |
| 4 | 3 | 35 | 8.05 |
| 5 | 3 | NULL | 8.46 |
| 6 | 1 | 54 | 51.86 |
| 7 | 3 | 2 | 21.08 |
| 8 | 3 | 27 | 11.13 |
| 9 | 2 | 14 | 30.07 |

**Step 2, WHERE `age IS NOT NULL`:** row 5 is removed. 9 rows left.

**Step 3, GROUP BY `pclass`:**

| pclass | rows | fares |
|---|---|---|
| 1 | 1, 3, 6 | 71.28, 53.10, 51.86 |
| 2 | 9 | 30.07 |
| 3 | 0, 2, 4, 7, 8 | 7.25, 7.93, 8.05, 21.08, 11.13 |

**Step 4, HAVING `COUNT(*) >= 2`:** class 2 has only 1 passenger, so the group is removed.

**Step 5, SELECT:**

| pclass | passengers | avg_fare |
|---|---|---|
| 1 | 3 | (71.28 + 53.10 + 51.86) / 3 = **58.75** |
| 3 | 5 | (7.25 + 7.93 + 8.05 + 21.08 + 11.13) / 5 = **11.09** |

**Step 6–7, ORDER BY `avg_fare DESC`:** class 1 first, then class 3.

**Step 8, LIMIT 1:** only class 1 remains. Now let's check each step with real queries.

In [11]:
con.execute('''
CREATE TABLE sample AS
SELECT rowid - 1 AS row, pclass, age, fare
FROM passengers
LIMIT 10
''')

print("Step 2, after WHERE:")
display(run("SELECT * FROM sample WHERE age IS NOT NULL"))

print("Step 3-5, groups before HAVING:")
display(run('''
SELECT pclass, COUNT(*) AS passengers, ROUND(AVG(fare), 2) AS avg_fare
FROM sample
WHERE age IS NOT NULL
GROUP BY pclass
'''))

Step 2, after WHERE:


,row,pclass,age,fare
0,0,3,22.0,7.2500
1,1,1,38.0,71.2833
2,2,3,26.0,7.9250
3,3,1,35.0,53.1000
4,4,3,35.0,8.0500
5,6,1,54.0,51.8625
6,7,3,2.0,21.0750
7,8,3,27.0,11.1333
8,9,2,14.0,30.0708


Step 3-5, groups before HAVING:


,pclass,passengers,avg_fare
0,1,3,58.75
1,2,1,30.07
2,3,5,11.09


In [12]:
print("Full query:")
run('''
SELECT pclass, COUNT(*) AS passengers, ROUND(AVG(fare), 2) AS avg_fare
FROM sample
WHERE age IS NOT NULL
GROUP BY pclass
HAVING COUNT(*) >= 2
ORDER BY avg_fare DESC
LIMIT 1
''')

Full query:


,pclass,passengers,avg_fare
0,1,3,58.75


> **Check:** each step matches the manual calculation: 3 groups before `HAVING`, then class 1 with 3 passengers and an average fare of 58.75.

Finally, the "WHERE cannot use COUNT" rule in action:

In [13]:
try:
    run("SELECT pclass, COUNT(*) FROM passengers WHERE COUNT(*) > 200 GROUP BY pclass")
except Exception as error:
    print("Error:", error)

Error: Execution failed on sql 'SELECT pclass, COUNT(*) FROM passengers WHERE COUNT(*) > 200 GROUP BY pclass': misuse of aggregate: COUNT()


The database refuses, because when `WHERE` runs (step 2) the groups don't exist yet. Moving the condition to `HAVING` fixes it.

---
## Summary

| Clause | Job | Remember |
|---|---|---|
| `SELECT` | choose and compute columns | `AS` gives an alias |
| `FROM` | choose the table | runs first |
| `WHERE` | filter rows | brackets when mixing `AND` and `OR` |
| `GROUP BY` | make groups | every selected column is grouped or aggregated |
| `HAVING` | filter groups | the place for conditions on `COUNT`, `SUM`, … |
| `ORDER BY`, `LIMIT` | sort, keep the top n | can use aliases |
| `NULL` | unknown value | `IS NULL`, `COUNT(*)` vs `COUNT(col)`, disappears from `<>` |
| Division | | whole ÷ whole = whole number: use `1.0 *` or `AVG` |

## What's next

Real databases split data into many tables. In notebook 02 we **join** tables together, on a music store database with 11 tables, and meet the join mistake that multiplied revenue by 9.

## References and further reading

- [SQLite documentation: SELECT](https://www.sqlite.org/lang_select.html) · [SQLite: NULL handling](https://www.sqlite.org/nulls.html)
- [PostgreSQL tutorial](https://www.postgresql.org/docs/current/tutorial-sql.html) (the same SQL on a server database)
- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html) · [pandas `read_sql_query`](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)